# Version v12a/c
* Change the predicted token from [CLS] to [FP2]

In [1]:
### LOGS 

# V1 LB 0.4428
## Token_clf : FB1 Models 7th ['cfg_ch_10c', 'cfg_ch_34d'] --> sub_01 (missing: 3 essay_ids)
## Effective : FB2 Model 5th single fold0 --> sub_02 


# V2 LB 0.4458 
## Token_clf : FB1 Models 7th ['cfg_ch_10c', 'cfg_ch_34d'] --> sub_01 (missing: 3 essay_ids)
## Effective : FB2 Model 5th single x 5 folds --> sub_02 

# V3 LB 0.411
# V2 + NN stacker / BUG?

# V4 LB 0.4x
# V2 + NN stacker 


# V5 LB 0.433
# V2 + LGB stacker 


# V6 LB 
## Token_clf : FB1 Model 1st Longformer 
## Effective : FB2 Model 5th single x 5 folds


# V7 LB 
## Token_clf : FB1 Model 1st Longformer + LGB
## Effective : FB2 Model 5th single x 5 folds



### TODO try other FB1 models as baseline 
# FP1 1st Longformer /+ LGB  
# - 
# - 


In [2]:
# !nvidia-smi

In [3]:
import shutil
from pathlib import Path

# transformers_path = Path('/opt/conda/lib/python3.7/site-packages/transformers')
# input_dir = Path('../input/deberta-v2-3-fast-tokenizer')

# convert_file = input_dir / 'convert_slow_tokenizer.py'
# conversion_path = transformers_path/convert_file.name
# # /opt/conda/lib/python3.7/site-packages/transformers/convert_slow_tokenizer.py

# if conversion_path.exists():
#     conversion_path.unlink()

# shutil.copy(convert_file, transformers_path)
# deberta_v2_path = transformers_path / 'models' / 'deberta_v2'

# for filename in ['tokenization_deberta_v2.py', 'tokenization_deberta_v2_fast.py', 'deberta__init__.py']:
#     if str(filename).startswith('deberta'):
#         filepath = deberta_v2_path/str(filename).replace('deberta', '')
#     else:
#         filepath = deberta_v2_path/filename
#     if filepath.exists():
#         filepath.unlink()

#     shutil.copy(input_dir/filename, filepath)

In [4]:
# !cat $conversion_path

In [5]:
%%capture

!pip install transformers==4.30

# Packages

In [6]:
import os, gc, pickle, math, time, random, copy, json
from glob import glob
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.metrics import log_loss
from scipy.special import softmax

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import Dataset

from torch.optim.swa_utils import AveragedModel, SWALR
# Mixed precision in Pytorch
from torch.cuda.amp import autocast, GradScaler

import transformers
from transformers import AutoConfig, AutoModel, AutoTokenizer
from transformers.models.deberta_v2.tokenization_deberta_v2_fast import DebertaV2TokenizerFast
from transformers import get_cosine_schedule_with_warmup, get_linear_schedule_with_warmup, AdamW


import warnings
warnings.filterwarnings('ignore')

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false' #'true'

print('torch', torch.__version__)
print('transformers', transformers.__version__)

torch 1.11.0
transformers 4.30.0


In [7]:
# from transformers.models.deberta_v2.tokenization_deberta_v2_fast import DebertaV2TokenizerFast
# from transformers.models.deberta.tokenization_deberta_fast import DebertaTokenizerFast 

# tokenizer = DebertaV2TokenizerFast.from_pretrained('microsoft/deberta-v3-large')

# Config 

In [8]:
class Config():
    debug = False
    use_cache_preds = False #True 
    
    competition_name = 'FeedbackPrize2'
    env = 'kaggle'
    mode = 'infer'    # 'infer', 'valid', 'infer_pl'
    seed = 1
    model_name = 'v12h'
    use_tqdm = True
    # For model
    if env == 'colab':
        backbone = 'microsoft/deberta-v3-large'
    elif env == 'kaggle':
        backbone = '../input/microsoftdebertav3large'
    
    tokenizer = DebertaV2TokenizerFast.from_pretrained(backbone)
    #tokenizer = AutoTokenizer.from_pretrained(backbone)
    config = AutoConfig.from_pretrained(backbone)
    config.output_hidden_states = True
    config.hidden_dropout_prob = 0.
    config.attention_probs_dropout_prob = 0.
    
    # Add new token
    cls_token = '[FP2]'
    cls_token_id = tokenizer.vocab_size + 1
    special_tokens_dict = {'additional_special_tokens': [cls_token]}
    tokenizer.add_special_tokens(special_tokens_dict)
    # For data
    discourse_type_map = {
        'Lead': 0,
        'Position': 1,
        'Claim': 2,
        'Counterclaim': 3,
        'Rebuttal': 4,
        'Evidence': 5,
        'Concluding Statement': 6,
    }
    label_map = {
        'Ineffective': 0,
        'Adequate': 1,
        'Effective': 2,
        'None': -100,
    }
    training_folds = [0, 1, 2, 3, 4]
    max_len = 1024
    batch_size = 16
    num_workers = os.cpu_count()
    # For training
    apex = False
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # Directories
    if env == 'colab':
        comp_data_dir = f'/content/drive/My Drive/Kaggle competitions/{competition_name}/comp_data'
        extra_data_dir = f'/content/drive/My Drive/Kaggle competitions/{competition_name}/extra_data'
        model_dir = f'/content/drive/My Drive/Kaggle competitions/{competition_name}/model'
        os.makedirs(os.path.join(model_dir, model_name.split('_')[0][:-1], model_name.split('_')[0][-1]), exist_ok = True)
    elif env == 'kaggle':
        comp_data_dir = '../input/feedback-prize-effectiveness'
        extra_data_dir = '../input/feedbackprize2extradata'
        model_dir = f'../input/feedbackpriz2{model_name}'
        


In [9]:
cfg = Config()

In [ ]:
VERSION = '6'

# Utils

In [10]:
def set_random_seed(seed, use_cuda = True):
    np.random.seed(seed) # cpu vars
    torch.manual_seed(seed) # cpu  vars
    random.seed(seed) # Python
    os.environ['PYTHONHASHSEED'] = str(seed) # Python hash building
    if use_cuda:
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # gpu vars
        torch.backends.cudnn.deterministic = True  #needed
        torch.backends.cudnn.benchmark = False

In [11]:
# ### FB1 mapping 

# int_to_class = {
#     "0": "Claim",
#     "1": "Concluding Statement",
#     "2": "Counterclaim",
#     "3": "Evidence",
#     "4": "Lead",
#     "5": "Position",
#     "6": "Rebuttal",
#     "7": "None"
# }

### FB2 mapping 
# discourse_type_map = {
#     'Lead': 0,
#     'Position': 1,
#     'Claim': 2,
#     'Counterclaim': 3,
#     'Rebuttal': 4,
#     'Evidence': 5,
#     'Concluding Statement': 6,
# }


### RATER mapping 
# 0 - Lead
# 1 - Position
# 2 - Claim
# 3 - Evidence
# 4 - Counterclaim
# 5 - Rebuttal
# 6 - Concluding Statement

discourse_type_to_int = {
    'Lead': 0,
    'Position': 1,
    'Claim': 2,
    'Evidence': 3, 
    'Counterclaim': 4, 
    'Rebuttal': 5, 
    'Concluding Statement': 6, 
}


In [12]:
print(cfg.discourse_type_map)
print(cfg.label_map)

{'Lead': 0, 'Position': 1, 'Claim': 2, 'Counterclaim': 3, 'Rebuttal': 4, 'Evidence': 5, 'Concluding Statement': 6}
{'Ineffective': 0, 'Adequate': 1, 'Effective': 2, 'None': -100}


# Import data

In [13]:
### Find missing ids 
test_rater = pd.read_csv('/kaggle/input/nlp-rater-utils/test.csv')
test_rater.columns = ['essay_id', 'text']
all_test_ids = test_rater.essay_id.unique()

len(all_test_ids)

10402

In [14]:
DEBUG = False # True 

if cfg.mode == 'infer':
    test_fp2 = pd.read_csv(os.path.join(cfg.comp_data_dir, 'test.csv'))
    test_fp2['discourse_effectiveness'] = 'None'
    test_fp2['fold'] = -1
    test_fp2[list(cfg.label_map.keys())[:3]] = -100
    display(test_fp2.head())

    #### FB1 7th
    #test = pd.read_csv('/kaggle/input/rater-fb2-infer-new-baseline/sub_01.csv')
    #test = test.rename(columns={'id': 'essay_id', 'full_text': 'text', 'class': 'discourse_type'})
    
    #### FB1 1st
    test = pd.read_csv('/kaggle/input/rater-fb1-infer-longformer-1st/sub01_fp2_longformer.csv')
    #test = pd.read_csv('/kaggle/input/rater-fb1-infer-longformer-lgb-1st/sub01_fp2_longformer_lgb.csv')
    
    if DEBUG: test = pd.read_csv('/kaggle/input/rater-infer-fb1/sub_01.csv')
    
    test['discourse_effectiveness'] = 'None'
    test['fold'] = -1
    test['discourse_id'] = range(len(test))
    test['discourse_id'] = test['discourse_id'].astype(str)
    ## ['Ineffective', 'Adequate', 'Effective']
    test['Ineffective'] = -100
    test['Adequate'] = -100
    test['Effective'] = -100
    ## rename cols 
    test = test.rename(columns={'essay_id_comp': 'essay_id'}) #'full_text': 'text', 'class': 'discourse_type'
    if ("text" or "full_text") not in test.columns:
        test = test.merge(test_rater, on='essay_id')

elif cfg.mode == 'infer_fp2':
    test = pd.read_csv(os.path.join(cfg.comp_data_dir, 'test.csv'))
    test['discourse_effectiveness'] = 'None'
    test['fold'] = -1
    test[list(cfg.label_map.keys())[:3]] = -100
    
elif cfg.mode == 'valid':
    test = pd.read_csv(os.path.join(cfg.extra_data_dir, 'train_seed_1.csv'))
    one_hot_label = np.zeros((test.shape[0], 3))
    one_hot_label[(np.arange(test.shape[0]), test['discourse_effectiveness'].map(cfg.label_map).values)] = 1
    test[list(cfg.label_map.keys())[:3]] = one_hot_label

elif cfg.mode == 'infer_pl':
    test = pd.read_csv(os.path.join(cfg.extra_data_dir, 'test_pl.csv'))
    test['discourse_id'] = test['discourse_id'].astype(str) + '_' + test['fold'].astype(str)
    test['essay_id'] = test['essay_id'].astype(str) + '_' + test['fold'].astype(str)

print('test', test.shape)
print('test ids', test.essay_id.nunique())
test.head()

,discourse_id,essay_id,discourse_text,discourse_type,discourse_effectiveness,fold,Ineffective,Adequate,Effective
0,a261b6e14276,D72CB1C11673,Making choices in life can be very difficult. ...,Lead,None,-1,-100,-100,-100
1,5a88900e7dc1,D72CB1C11673,Seeking multiple opinions can help a person ma...,Position,None,-1,-100,-100,-100
2,9790d835736b,D72CB1C11673,it can decrease stress levels,Claim,None,-1,-100,-100,-100
3,75ce6d68b67b,D72CB1C11673,a great chance to learn something new,Claim,None,-1,-100,-100,-100
4,93578d946723,D72CB1C11673,can be very helpful and beneficial.,Claim,None,-1,-100,-100,-100


test (90974, 14)
test ids 10402


,Unnamed: 0,essay_id,class,predictionstring,discourse_type,score_discourse_effectiveness_0,score_discourse_effectiveness_1,discourse_effectiveness,fold,discourse_id,Ineffective,Adequate,Effective,text
0,0,629862A6AB15,Claim,20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 3...,2,-100,-100,None,-1,0,-100,-100,-100,i think cell phone should be brought to school...
1,1,629862A6AB15,Claim,39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 5...,2,-100,-100,None,-1,1,-100,-100,-100,i think cell phone should be brought to school...
2,2,629862A6AB15,Evidence,62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 7...,3,-100,-100,None,-1,2,-100,-100,-100,i think cell phone should be brought to school...
3,3,629862A6AB15,Evidence,84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 9...,3,-100,-100,None,-1,3,-100,-100,-100,i think cell phone should be brought to school...
4,4,629862A6AB15,Position,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19,1,-100,-100,None,-1,4,-100,-100,-100,i think cell phone should be brought to school...


In [15]:
def predstring_to_text(df):
    
    df_out = df.copy()
    #convert predictionstring to list of integers
    df_out['predictionstring_int'] = df_out['predictionstring'].apply(lambda x: [int(float(y)) for y in x.split(' ')])

    texts = []
    for idx, row in df_out.iterrows():
        start, end = row.predictionstring_int[0], row.predictionstring_int[-1]
        texts.append(row.text[start:end])
        # break
    df_out['discourse_text'] = texts
    del df_out['predictionstring_int']
    return df_out

# extract discourse_text 
if "discourse_text" not in test.columns:
    #preddf = preddf.merge(test_data, on='id', how='left')
    test = predstring_to_text(test)

In [16]:
# test_fp2.shape
test.head()

,Unnamed: 0,essay_id,class,predictionstring,discourse_type,score_discourse_effectiveness_0,score_discourse_effectiveness_1,discourse_effectiveness,fold,discourse_id,Ineffective,Adequate,Effective,text,discourse_text
0,0,629862A6AB15,Claim,20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 3...,2,-100,-100,None,-1,0,-100,-100,-100,i think cell phone should be brought to school...,hould be brought t
1,1,629862A6AB15,Claim,39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 5...,2,-100,-100,None,-1,1,-100,-100,-100,i think cell phone should be brought to school...,school and allowed to
2,2,629862A6AB15,Evidence,62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 7...,3,-100,-100,None,-1,2,-100,-100,-100,i think cell phone should be brought to school...,be used during lunch
3,3,629862A6AB15,Evidence,84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 9...,3,-100,-100,None,-1,3,-100,-100,-100,i think cell phone should be brought to school...,eriods and free times because if we couldn't s...
4,4,629862A6AB15,Position,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19,1,-100,-100,None,-1,4,-100,-100,-100,i think cell phone should be brought to school...,i think cell phone


In [17]:
discourse_map = dict(zip(test['discourse_id'].astype(str), test.index))
inv_discourse_map = {v: k for k, v in discourse_map.items()}

In [18]:
# Read each essay
if cfg.mode == 'infer':
    essay_paths = glob(f'{cfg.comp_data_dir}/test/*.txt')
    essay_dict = {
        'essay_id': [i.split('/')[-1].split('.')[0] for i in tqdm(essay_paths)],
        'text': [open(i).read() for i in tqdm(essay_paths)],
    }
    essay_df = pd.DataFrame.from_dict(essay_dict)
    assert essay_df.shape[0] == test_fp2.essay_id.nunique()
    display(essay_df)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

,essay_id,text
0,D72CB1C11673,Making choices in life can be very difficult. ...


In [19]:
if cfg.mode == 'infer':
    test_fp2 = test_fp2.merge(essay_df, right_on = 'essay_id', left_on = 'essay_id')

test_fp2.head()

,discourse_id,essay_id,discourse_text,discourse_type,discourse_effectiveness,fold,Ineffective,Adequate,Effective,text
0,a261b6e14276,D72CB1C11673,Making choices in life can be very difficult. ...,Lead,None,-1,-100,-100,-100,Making choices in life can be very difficult. ...
1,5a88900e7dc1,D72CB1C11673,Seeking multiple opinions can help a person ma...,Position,None,-1,-100,-100,-100,Making choices in life can be very difficult. ...
2,9790d835736b,D72CB1C11673,it can decrease stress levels,Claim,None,-1,-100,-100,-100,Making choices in life can be very difficult. ...
3,75ce6d68b67b,D72CB1C11673,a great chance to learn something new,Claim,None,-1,-100,-100,-100,Making choices in life can be very difficult. ...
4,93578d946723,D72CB1C11673,can be very helpful and beneficial.,Claim,None,-1,-100,-100,-100,Making choices in life can be very difficult. ...


In [20]:
### RATER test data formated 

test = test[test_fp2.columns.tolist()+['predictionstring', 'class']]
test.head()

,discourse_id,essay_id,discourse_text,discourse_type,discourse_effectiveness,fold,Ineffective,Adequate,Effective,text,predictionstring,class
0,0,629862A6AB15,hould be brought t,2,None,-1,-100,-100,-100,i think cell phone should be brought to school...,20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 3...,Claim
1,1,629862A6AB15,school and allowed to,2,None,-1,-100,-100,-100,i think cell phone should be brought to school...,39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 5...,Claim
2,2,629862A6AB15,be used during lunch,3,None,-1,-100,-100,-100,i think cell phone should be brought to school...,62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 7...,Evidence
3,3,629862A6AB15,eriods and free times because if we couldn't s...,3,None,-1,-100,-100,-100,i think cell phone should be brought to school...,84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 9...,Evidence
4,4,629862A6AB15,i think cell phone,1,None,-1,-100,-100,-100,i think cell phone should be brought to school...,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19,Position


In [21]:
test.essay_id.nunique()

10402

In [22]:
missing_ids = list(set(all_test_ids) - set(test.essay_id))
missing_ids

[]

In [23]:
if len(missing_ids)>0:
    for mid in missing_ids:
        print(mid)
        print(test_rater[test_rater.essay_id_comp==mid]['full_text'].values[0].rstrip())
        print()

In [24]:
del test_rater
gc.collect()

102

# Reformulate the train data so that entire a text is processed at a whole

In [25]:
def organize_df(df, idx, cfg):
    discourse_ids = '|'.join(df['discourse_id'].astype(str).tolist())
    #discourses = ''.join((cfg.cls_token + df['discourse_type'] + '. ' + df['discourse_text']).tolist())
    discourses = ''.join((cfg.cls_token + df['class'] + '. ' + df['discourse_text']).tolist())
    label_list = '|'.join(df[list(cfg.label_map.keys())[:3]].values.reshape(-1).astype(str).tolist())
    label_name_list = '|'.join(df['discourse_effectiveness'].tolist())
    fold = df['fold'].unique()[0]
    return pd.DataFrame({
        'discourse_ids': discourse_ids,
        'discourses': discourses,
        'label_list': label_list,
        'label_name_list': label_name_list,
        'fold': fold,
    }, index = [idx])

In [26]:
test_df = []
for idx, df_ in tqdm(test.groupby('essay_id')):
    test_df.append(organize_df(df_, idx, cfg))
test_df = pd.concat(test_df).reset_index()
test.set_index('discourse_id', inplace = True)

print('test_df', test_df.shape)
test_df.head()

  0%|          | 0/10402 [00:00<?, ?it/s]

test_df (10402, 6)


,index,discourse_ids,discourses,label_list,label_name_list,fold
0,0006ED03C701,20395|20396|20397|20398|20399|20400,[FP2]Claim. ience adventure [FP2]Evidence. ion...,-100|-100|-100|-100|-100|-100|-100|-100|-100|-...,None|None|None|None|None|None,-1
1,000A58BC095E,13935|13936|13937|13938|13939,[FP2]Claim. gs get back on their fe[FP2]Eviden...,-100|-100|-100|-100|-100|-100|-100|-100|-100|-...,None|None|None|None|None,-1
2,000BAD50D026,38614|38615|38616|38617|38618|38619|38620,[FP2]Claim. ould benefit from being able atten...,-100|-100|-100|-100|-100|-100|-100|-100|-100|-...,None|None|None|None|None|None|None,-1
3,001C3ABFA993,5013|5014|5015|5016|5017|5018|5019,[FP2]Claim. reading facial expressions can[FP2...,-100|-100|-100|-100|-100|-100|-100|-100|-100|-...,None|None|None|None|None|None|None,-1
4,002F3E0E697E,28722|28723|28724|28725|28726|28727|28728|2872...,"[FP2]Claim. ves"" article [FP2]Claim. ures he w...",-100|-100|-100|-100|-100|-100|-100|-100|-100|-...,None|None|None|None|None|None|None|None|None,-1


# Dataset

In [27]:
def create_label(cfg, input_ids, overflow_to_sample_mapping, label_list, discourse_id_list):
    discourse_ids = []
    labels = []
    is_tail = []
    for i, (input_id, sample_mapping) in enumerate(zip(input_ids, overflow_to_sample_mapping)):
        input_ids_array = np.array(input_id)
        label = np.zeros((len(input_id), 3)) - 100
        discourse_id = np.zeros_like(input_id) - 100
        current_label = label_list[sample_mapping]
        # Recover the label
        current_label = np.array(current_label.split('|')).astype(float).reshape(-1, 3)
        if i > 0:
            past_sample_mapping = overflow_to_sample_mapping[i-1]
        else:
            past_sample_mapping = -1

        num_cls_tokens = sum(input_ids_array == cfg.cls_token_id)
        if sample_mapping == past_sample_mapping:
            start = old_end
            is_tail.append(1)
        else:
            start = 0
            is_tail.append(0)
        
        end = start + num_cls_tokens
        old_end = end
        
        chosen_tokens = input_ids_array == cfg.cls_token_id
        label[chosen_tokens] = current_label[start:end]
        discourse_id[chosen_tokens] = discourse_id_list[sample_mapping][start:end]
        labels.append(label)
        discourse_ids.append(discourse_id)

    return labels, discourse_ids, is_tail

def prepare_dataset(example, cfg = cfg, mode = 'train'):
    discourses = example['discourses']
    label_list = example['label_list']
    discourse_id_list = [[discourse_map[i] for i in l.split('|')] for l in example['discourse_ids']]
    
    if mode == 'train' or mode == 'valid':
        padding = 'max_length'
    else:
        padding = False

    tokenized_discourses = cfg.tokenizer(
        discourses, 
         return_attention_mask = True, 
         truncation = True, 
         max_length = cfg.max_len, 
         padding = padding,
         return_length = True, 
         return_overflowing_tokens = True,
        #return_offsets_mapping=True, 

        )
    ###V2: ['overflowing_tokens', 'num_truncated_tokens', 'input_ids', 'token_type_ids', 'length', 'attention_mask']
    ###V2 fast: ['input_ids', 'token_type_ids', 'attention_mask', 'length', 'overflow_to_sample_mapping']
    #print(tokenized_discourses.keys())
    #print(tokenized_discourses)
    
    input_ids = tokenized_discourses['input_ids']
    overflow_to_sample_mapping = tokenized_discourses['overflow_to_sample_mapping']
    #overflow_tokens = tokenized_discourses['overflowing_tokens']
    assert max(overflow_to_sample_mapping) == len(label_list) - 1
    
    labels, discourse_ids, is_tail = create_label(cfg, input_ids, overflow_to_sample_mapping, label_list, discourse_id_list)
    tokenized_discourses['labels'] = labels
    tokenized_discourses['discourse_ids'] = discourse_ids
    tokenized_discourses['is_tail'] = is_tail

    return tokenized_discourses

def generate_dataset(cfg, df, mode = 'train'):
    ds = Dataset.from_pandas(df)
    tokenized_ds = ds.map(prepare_dataset, fn_kwargs = {'cfg': cfg, 'mode': mode}, batched = True, batch_size = 20_000, remove_columns = ds.column_names)
    if mode == 'infer':
        # Sort dataset
        len_sortidx = np.argsort(tokenized_ds['length'])
        len_rank = np.vectorize(lambda x: len_sortidx.tolist().index(x))(np.arange(len(len_sortidx)))
        sorted_tokenized_ds = tokenized_ds.sort('length')
    else:
        sorted_tokenized_ds, len_rank = None, None
    return tokenized_ds, sorted_tokenized_ds, len_rank

In [28]:
class FeedbackPrize2_Dataset(Dataset):
    def __init__(self, cfg, df, mode = 'train'):
        dataset, sorted_dataset, len_rank = generate_dataset(cfg, df, mode = mode)
        self.dataset = dataset
        self.mode = mode
        if mode == 'infer':
            self.dataset = sorted_dataset
            self.len_rank = len_rank
    
    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        if self.mode == 'infer':
            item['len_rank'] = self.len_rank[idx]
        else:
            item['len_rank'] = None
        return item

In [29]:
from typing import Any, Dict, Iterable, Iterator, List, Optional, Tuple, Union

def pad_sequence(sequences: List[List[int]], max_len, padding_side = 'right', dim = 1):
    if padding_side == 'right':
        if dim > 1:
            return [x + [[-100] * dim] * (max_len - len(x)) for x in sequences]
        else:
            return [x + [-100] * (max_len - len(x)) for x in sequences]
    else:
        if dim > 1:
            return [[[-100] * dim] * (max_len - len(x)) + x for x in sequences]
        else:    
            return [[-100] * (max_len - len(x)) + x for x in sequences]

In [30]:
def collate_fn(batch):
    input_ids = []
    attention_mask = []
    token_type_ids = []
    is_tail = []
    labels = []
    discourse_ids = []
    len_rank = []
    
    max_len = max([len(item['input_ids']) for item in batch])

    for item in batch:
        batch_input_ids = item['input_ids']
        batch_attention_mask = item['attention_mask']
        batch_token_type_ids = item['token_type_ids']
        batch_is_tail = item['is_tail']
        batch_labels = item['labels']
        batch_discourse_ids = item['discourse_ids']

        input_ids.append(batch_input_ids)
        attention_mask.append(batch_attention_mask)
        token_type_ids.append(batch_token_type_ids)
        is_tail.append(batch_is_tail)
        labels.append(batch_labels)
        discourse_ids.append(batch_discourse_ids)

        if item['len_rank'] is not None:
            batch_len_rank = item['len_rank']
            len_rank.append(batch_len_rank)
        else:
            len_rank.append(-1)
        
    input_dict = {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'token_type_ids': token_type_ids,
    }
    
    # Pad the inputs
    input_dict = cfg.tokenizer.pad(input_dict)
    labels = pad_sequence(labels, max_len = max_len, dim = 3)
    discourse_ids = pad_sequence(discourse_ids, max_len = max_len)
    
    return {
        'input_ids': torch.tensor(input_dict['input_ids'], dtype = torch.long),
        'attention_mask': torch.tensor(input_dict['attention_mask'], dtype = torch.long),
        'token_type_ids': torch.tensor(input_dict['token_type_ids'], dtype = torch.long),
        'is_tail': torch.tensor(is_tail, dtype = torch.long),
        'labels': torch.tensor(labels, dtype = torch.float),
        'discourse_ids': torch.tensor(discourse_ids, dtype = torch.long),
        'len_rank': torch.tensor(len_rank, dtype = torch.long),
    }

In [31]:
dataset = FeedbackPrize2_Dataset(cfg, test_df, mode='infer')
dataloader = DataLoader(dataset, batch_size = 4, num_workers = 2, shuffle = False, collate_fn = collate_fn)

print(len(dataset))

  0%|          | 0/1 [00:00<?, ?ba/s]

10402


In [32]:
item = next(iter(dataloader))
item

You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'input_ids': tensor([[     1, 128001,  18172,    260,  61009,   2976,  37655, 128001,   8380,
             260,    307,   5274,    274,    431,    428,    272,   2976,      2,
               0,      0,      0,      0],
         [     1, 128001,  21049,    260,  20934,    281,    333,    547, 128001,
           21049,    260,    266,   1399,    688,   1789,    262,   1376,      2,
               0,      0,      0,      0],
         [     1, 128001,  12993,    260,    299,    282,    379,   2136,    267,
             324, 128001,  12993,    260,   1931,    604,    634,    264,    424,
             347,   2985,      2,      0],
         [     1, 128001,  21049,    260,   5173,    263,   2976, 128001,  12993,
             260,    961,    275,   2164,  21494,   6201, 128001,   8380,    260,
             488,    264,   1798,      2]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
         [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

# Model

In [33]:
class FeedbackPrize2_Model(nn.Module):
    def __init__(self, cfg):
        super(FeedbackPrize2_Model, self).__init__()
        self.cfg = cfg
        # Backbone
        self.backbone = AutoModel.from_pretrained(cfg.backbone, config = cfg.config, ignore_mismatched_sizes = True)
        self.backbone.resize_token_embeddings(len(cfg.tokenizer))

        # Multidropout
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.2)
        self.dropout3 = nn.Dropout(0.3)
        self.dropout4 = nn.Dropout(0.4)
        self.dropout5 = nn.Dropout(0.5)
        
        # GRU
        self.rnn = nn.GRU(input_size = cfg.config.hidden_size, hidden_size = cfg.config.hidden_size // 2,
                          bidirectional = True, batch_first = True, dropout = 0.1, num_layers = 1)
        
        # Head
        self.head = nn.Linear(cfg.config.hidden_size, 3)
        
        self._init_weights(self.head)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean = 0.0, std = cfg.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean = 0.0, std = cfg.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

    def criterion(self, pred, true):
        loss = nn.CrossEntropyLoss(ignore_index = -100)(pred.permute(0, 2, 1), true)
        return loss

    def forward(self, input_ids, attention_mask, token_type_ids, label = None):
        output_backbone = self.backbone(input_ids = input_ids, attention_mask = attention_mask, token_type_ids = token_type_ids).last_hidden_state    # batch_size, seq_len, hidden_size
        output_backbone = self.rnn(output_backbone)[0]
        
        output1 = self.head(self.dropout1(output_backbone))
        output2 = self.head(self.dropout2(output_backbone))
        output3 = self.head(self.dropout3(output_backbone))
        output4 = self.head(self.dropout4(output_backbone))
        output5 = self.head(self.dropout5(output_backbone))

        output = (output1 + output2 + output3 + output4 + output5) / 5

        if label is not None:
            loss = self.criterion(output, label)
        else:
            loss = None
        _PADDING_VALUE = -1e30 if output.dtype == torch.float32 else -1e4
        return loss, F.pad(output, (0, 0, 0, self.cfg.max_len - output.shape[1]), 'constant', _PADDING_VALUE)

# Inferring functions

In [34]:
def metric(y_pred, y_true):
    return log_loss(y_true, y_pred)


def infer_fn(cfg, model, infer_dataloader):
    # Set up for training
    model.eval()

    preds = []
    all_input_ids = []
    all_is_tails = []
    all_len_rank = []
    all_discourse_ids = []

    if cfg.use_tqdm:
        tbar = tqdm(infer_dataloader)
    else:
        tbar = infer_dataloader

    for i, item in enumerate(tbar):
        # Set up inputs
        input_ids = item['input_ids'].to(cfg.device)
        attention_mask = item['attention_mask'].to(cfg.device)
        token_type_ids = item['token_type_ids'].to(cfg.device)
        is_tail = item['is_tail']
        len_rank = item['len_rank']
        discourse_ids = item['discourse_ids']

        batch_size = input_ids.shape[0]

        # Forward
        with torch.no_grad():
            with autocast(enabled = cfg.apex):
                _, batch_pred = model(input_ids, attention_mask, token_type_ids)

        # Pad also the input_ids
        input_ids = F.pad(input_ids, (0, cfg.max_len - input_ids.shape[1]), 'constant', cfg.tokenizer.pad_token_id)
        discourse_ids = F.pad(discourse_ids, (0, cfg.max_len - discourse_ids.shape[1]), 'constant', -100)

        # Store the predictions
        preds.append(batch_pred.detach().cpu().numpy())
        all_input_ids.append(input_ids.detach().cpu().numpy())
        all_is_tails.append(is_tail.numpy())
        all_len_rank.append(len_rank.numpy())
        all_discourse_ids.append(discourse_ids.numpy())
    
    # Concatenate batch materials
    preds = np.concatenate(preds, axis = 0)
    all_input_ids = np.concatenate(all_input_ids, axis = 0)
    all_is_tails = np.concatenate(all_is_tails, axis = 0)
    all_len_rank = np.concatenate(all_len_rank, axis = 0)
    all_discourse_ids = np.concatenate(all_discourse_ids, axis = 0)
    
    # Re-order the sorted predictions - size: (num of essays, max length, 3)
    preds = preds[all_len_rank]
    all_input_ids = all_input_ids[all_len_rank]
    all_is_tails = all_is_tails[all_len_rank]
    all_discourse_ids = all_discourse_ids[all_len_rank]
    
    # Get predictions at the CLS tokens only, going from essays to discourses
    preds = preds[np.where(all_input_ids == cfg.cls_token_id)]
    discourse_ids = all_discourse_ids[np.where(all_input_ids == cfg.cls_token_id)]

    return preds.astype(np.float64), np.array([inv_discourse_map[i] for i in discourse_ids])

In [35]:
def infer_loop(cfg, infer_dataloader, old_df = None, new_df = None, fold = 0):
    print(f' Infer Fold {fold} '.center(50, '*'))
    set_random_seed(cfg.seed + fold)
    
    model = FeedbackPrize2_Model(cfg).to(cfg.device)
    model_path = os.path.join(cfg.model_dir, f'fold_{fold}.pt')
    print(f'Loading the pre-trained model from {model_path}...')
    ckp = torch.load(model_path, map_location = cfg.device)
    try:
        model.load_state_dict(ckp['state_dict'])
        pred, discourse_ids = infer_fn(cfg, model, infer_dataloader)
    except:
        swa_model = AveragedModel(model)
        swa_model.load_state_dict(ckp['state_dict'])
        pred, discourse_ids = infer_fn(cfg, swa_model, infer_dataloader)
    
    if cfg.mode == 'valid':
        assert (old_df is not None) and (new_df is not None), "The arguments 'old_df' and 'new_df' must be given in the validation mode!"
        true = np.array([cfg.label_map[item] for sublist in [i.split('|') for i in new_df['label_name_list'].tolist()] for item in sublist])
        discourse_ids = np.array([item for sublist in [i.split('|') for i in new_df['discourse_ids'].tolist()] for item in sublist])
        old_df.loc[discourse_ids, list(cfg.label_map.keys())[:3]] = softmax(pred, axis = -1)
        
        # Scoring
        best_score = metric(softmax(pred, axis = -1), true)
        print(f'Score: {best_score}')
    elif cfg.mode == 'infer_pl':
        discourse_ids = np.array([item for sublist in [i.split('|') for i in new_df['discourse_ids'].tolist()] for item in sublist])
        old_df.loc[discourse_ids, list(cfg.label_map.keys())[:3]] = softmax(pred, axis = -1)
        
    del model, ckp
    torch.cuda.empty_cache()
    gc.collect()
    
    return pred, old_df, discourse_ids

# Inference FB2

In [36]:
# def main(): 
if 1:
    if cfg.mode == 'infer':
        print('Preparing the inferring dataloader...')
        test_dataset = FeedbackPrize2_Dataset(cfg, test_df, mode = 'infer')
        test_dataloader = DataLoader(test_dataset, batch_size = cfg.batch_size, num_workers = cfg.num_workers, shuffle = False, collate_fn = collate_fn)
    
    oofs = []
    for i, fold in enumerate(cfg.training_folds):
        if cfg.mode == 'valid' or cfg.mode == 'infer_pl':
            print(f'Preparing the dataloader for fold {fold}...')
            val = test[test.fold == fold]
            val_df = test_df[test_df.fold == fold]
            
            test_dataset = FeedbackPrize2_Dataset(cfg, val_df, mode = 'infer')
            test_dataloader = DataLoader(test_dataset, batch_size = cfg.batch_size, num_workers = cfg.num_workers, shuffle = False, collate_fn = collate_fn)
            
        else:
            val = None
            val_df = None
            
        fold_pred, val, discourse_ids = infer_loop(cfg, test_dataloader, old_df=val, new_df=val_df, fold=fold)
        
        if cfg.mode == 'infer':
            if i == 0:
                pred = fold_pred / len(cfg.training_folds)
            else:
                pred += fold_pred / len(cfg.training_folds)
        
        if cfg.mode == 'infer_pl':
            val.rename(columns = {'index': 'discourse_id'}).to_csv(os.path.join(cfg.extra_data_dir, f'train_pl_{fold}.csv'), index = False)

        oofs.append(val)
    #####
    preds_softmax = softmax(pred, axis=-1)
    print('preds_softmax', preds_softmax.shape)
    np.save('preds_softmax.npy', preds_softmax)
    
    if cfg.mode == 'valid':
        oofs = pd.concat(oofs).loc[test.index]
        pred = oofs[list(cfg.label_map.keys())[:3]].values
        true = oofs['discourse_effectiveness'].map(cfg.label_map).values
        # Scoring
        score = metric(pred, true)
        print('*' * 50)
        print(f'OOF score: {score}')
        oofs = oofs.reset_index()
        display(oofs.head())
        oofs.rename(columns = {'index': 'discourse_id'}).to_pickle(os.path.join(cfg.model_dir, cfg.model_name.split('_')[0][:-1], cfg.model_name.split('_')[0][-1], f'oof_01234.pkl'))
    
    elif cfg.mode == 'infer':
        try:
            test.loc[discourse_ids, ['Ineffective', 'Adequate', 'Effective']] = softmax(pred, axis=-1)
            #test.loc[[int(i) for i in discourse_ids], ['Ineffective', 'Adequate', 'Effective']] = preds_softmax  
            display(test.head())
            test.to_csv('sub_02.csv', index=False)
        except:
            pass 
        
        ####
        #submission = pd.read_csv(os.path.join(cfg.comp_data_dir, 'sample_submission.csv'))
        #submission[['Ineffective', 'Adequate', 'Effective']] = test.loc[submission.discourse_id, ['Ineffective', 'Adequate', 'Effective']].values
        #submission[['discourse_id', 'Ineffective', 'Adequate', 'Effective']].to_csv('submission.csv', index = False)
        #display(submission.head())
        #submission.to_csv('submission.csv', index = False)

In [37]:
cfg.use_cache_preds

False

In [38]:
# if __name__ == '__main__':
if cfg.use_cache_preds:
    ### USE cached predictions 
    !cp -r /kaggle/input/rater-feedbackprize2-infer/sub_02.csv ./sub_02.csv
    !cp -r /kaggle/input/rater-feedbackprize2-infer/preds_softmax.npy ./preds_softmax.npy

else:
    main()

Preparing the inferring dataloader...


  0%|          | 0/1 [00:00<?, ?ba/s]

****************** Infer Fold 0 ******************


Some weights of the model checkpoint at ../input/microsoftdebertav3large were not used when initializing DebertaV2Model: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing DebertaV2Model from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaV2Model from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Loading the pre-trained model from ../input/feedbackpriz2v12h/fold_0.pt...


...

Loading the pre-trained model from ../input/feedbackpriz2v12h/fold_4.pt...


  0%|          | 0/651 [00:00<?, ?it/s]

You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


preds_softmax (90974, 3)


KeyError: "None of [Int64Index([20395, 20396, 20397, 20398, 20399, 20400, 13935, 13936, 13937,\n            13938,\n            ...\n            24428,  9587,  9588,  9589,  9590,  9591,  9592,  9593,  9594,\n             9595],\n           dtype='int64', name='discourse_id', length=90974)] are in the [index]"

In [40]:
preds_softmax = np.load('preds_softmax.npy')
print(preds_softmax.shape)

preds_df = pd.DataFrame(preds_softmax, columns=list(cfg.label_map.keys())[:3])
preds_df

(90974, 3)


,Ineffective,Adequate,Effective
0,0.275358,0.717731,0.006911
1,0.533104,0.464270,0.002625
2,0.510031,0.486353,0.003616
3,0.442183,0.555225,0.002592
4,0.324078,0.673077,0.002845
...,...,...,...
90969,0.463571,0.531671,0.004758
90970,0.292646,0.700104,0.007250
90971,0.461917,0.534233,0.003850
90972,0.438243,0.558494,0.003263


In [70]:
# test.loc[test.index, ['Ineffective', 'Adequate', 'Effective']] = preds_softmax
test.to_csv('sub_02.csv') 
test.head()

,essay_id,discourse_text,discourse_type,discourse_effectiveness,fold,Ineffective,Adequate,Effective,text,predictionstring,class
discourse_id,,,,,,,,,,,
0,629862A6AB15,hould be brought t,2,None,-1,0.275358,0.717731,0.006911,i think cell phone should be brought to school...,20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 3...,Claim
1,629862A6AB15,school and allowed to,2,None,-1,0.533104,0.464270,0.002625,i think cell phone should be brought to school...,39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 5...,Claim
2,629862A6AB15,be used during lunch,3,None,-1,0.510031,0.486353,0.003616,i think cell phone should be brought to school...,62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 7...,Evidence
3,629862A6AB15,eriods and free times because if we couldn't s...,3,None,-1,0.442183,0.555225,0.002592,i think cell phone should be brought to school...,84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 9...,Evidence
4,629862A6AB15,i think cell phone,1,None,-1,0.324078,0.673077,0.002845,i think cell phone should be brought to school...,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19,Position


# Submission

In [45]:
sample_sub = pd.read_csv('/kaggle/input/nlp-rater-utils/sample_submission.csv')
sample_sub

,essay_id_comp,predictionstring,score_discourse_effectiveness_0,score_discourse_effectiveness_1,discourse_type
0,215B5CA132E4,1 2 3 4 5 6 7 8 9 10 11,0.483,0.517,0
1,215B5CA132E4,12 13 14 15 16 17 18 19 20 21 22,0.217,0.783,1
2,215B5CA132E4,23 24 25 26 27 28 29 30 31 32 33 34 35 36,0.521,0.479,3
3,215B5CA132E4,35 36 37 38 39 40 41 42 43 44 45,0.359,0.641,2


In [51]:
sub2 = pd.read_csv('sub_02.csv')
sub2 = sub2.rename(columns={'essay_id': 'essay_id_comp'})
## token types
# sub2['discourse_type'] = sub2['discourse_type'].map(discourse_type_to_int) 

### convert predstring to str 
# sub2['predictionstring'] = sub2.apply(lambda x: " ".join(str(x) for x in eval(x.predictionstring)), axis=1)

## effectiveness 
sub2['score_discourse_effectiveness_0'] = preds_df['Ineffective'].values  
sub2['score_discourse_effectiveness_1'] = preds_df['Adequate'].values + preds_df['Effective'].values  

In [52]:
sub2 = sub2[sample_sub.columns]
sub2.to_csv(f'submission_v{VERSION}.csv')

In [53]:
sub2.essay_id_comp.nunique()

10402

In [54]:
sub2

,essay_id_comp,predictionstring,score_discourse_effectiveness_0,score_discourse_effectiveness_1,discourse_type
0,629862A6AB15,20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 3...,0.275358,0.724642,2
1,629862A6AB15,39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 5...,0.533104,0.466896,2
2,629862A6AB15,62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 7...,0.510031,0.489969,3
3,629862A6AB15,84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 9...,0.442183,0.557817,3
4,629862A6AB15,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19,0.324078,0.675922,1
...,...,...,...,...,...
90969,9FDFDC5EB16E,174 175 176 177 178 179 180 181 182 183 184 18...,0.463571,0.536429,3
90970,9FDFDC5EB16E,607 608 609 610 611 612 613 614 615 616 617 61...,0.292646,0.707354,3
90971,9FDFDC5EB16E,887 888 889 890 891 892 893 894 895 896 897 89...,0.461917,0.538083,3
90972,9FDFDC5EB16E,1166 1167 1168 1169 1170 1171 1172 1173 1174 1...,0.438243,0.561757,6


In [ ]:
# ### temp hack - add dummy preds for missing ids --> TODO debug 

# if len(missing_ids)>0:
#     dummy_df = []
#     for i, mid in enumerate(missing_ids):
#         tmp = {
#             'essay_id_comp': mid,
#             'predictionstring': ['0 1 2 3 4 5 6 7 8 9'],
#             'score_discourse_effectiveness_0': 0.95,
#             'score_discourse_effectiveness_1': 0.05,
#             'discourse_type': 0,
#         }
#         dummy_df.append(pd.DataFrame(tmp))
#     dummy_df = pd.concat(dummy_df, ignore_index=True) 
#     ##
#     sub_rater = pd.concat([sub2[sample_sub.columns], dummy_df], axis=0)
#     sub_rater = sub_rater.reset_index(drop=True)

In [55]:
assert sub2.essay_id_comp.nunique() == len(all_test_ids)
sub2.essay_id_comp.nunique()

10402

In [56]:
sample_sub.dtypes

essay_id_comp                       object
predictionstring                    object
score_discourse_effectiveness_0    float64
score_discourse_effectiveness_1    float64
discourse_type                       int64
dtype: object

In [57]:
sub2.dtypes

essay_id_comp                       object
predictionstring                    object
score_discourse_effectiveness_0    float64
score_discourse_effectiveness_1    float64
discourse_type                       int64
dtype: object

# Stacker NN

In [71]:
sub = pd.read_csv('sub_02.csv')
sub = sub.rename(columns={'essay_id': 'essay_id_comp'})

In [73]:
# pp = preds_softmax.copy()
# orig_preds = pp.copy()
# df = pd.read_csv('sub_02.csv')

df = sub.copy()
df = df.rename(columns={'essay_id_comp': 'essay_id'})
df.shape

(90974, 12)

In [74]:
df.head()

,discourse_id,essay_id,discourse_text,discourse_type,discourse_effectiveness,fold,Ineffective,Adequate,Effective,text,predictionstring,class
0,0,629862A6AB15,hould be brought t,2,None,-1,0.275358,0.717731,0.006911,i think cell phone should be brought to school...,20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 3...,Claim
1,1,629862A6AB15,school and allowed to,2,None,-1,0.533104,0.464270,0.002625,i think cell phone should be brought to school...,39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 5...,Claim
2,2,629862A6AB15,be used during lunch,3,None,-1,0.510031,0.486353,0.003616,i think cell phone should be brought to school...,62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 7...,Evidence
3,3,629862A6AB15,eriods and free times because if we couldn't s...,3,None,-1,0.442183,0.555225,0.002592,i think cell phone should be brought to school...,84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 9...,Evidence
4,4,629862A6AB15,i think cell phone,1,None,-1,0.324078,0.673077,0.002845,i think cell phone should be brought to school...,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19,Position


In [75]:
preds_df.head()

,Ineffective,Adequate,Effective
0,0.275358,0.717731,0.006911
1,0.533104,0.464270,0.002625
2,0.510031,0.486353,0.003616
3,0.442183,0.555225,0.002592
4,0.324078,0.673077,0.002845


In [76]:
label_cols = ["Adequate", "Effective", "Ineffective"]
# label_cols = ["Ineffective", "Adequate", "Effective"]

oof_cols = []
for j, l in enumerate(label_cols):
    #df[f"oof_{l}"] = pp[:,j] # ORIGINAL CLASS ORDER 
    df[f"oof_{l}"] = preds_df[l]  # NEW CLASS ORDER 
    oof_cols.append(f"oof_{l}")

In [77]:
df.head(3)

,discourse_id,essay_id,discourse_text,discourse_type,discourse_effectiveness,fold,Ineffective,Adequate,Effective,text,predictionstring,class,oof_Adequate,oof_Effective,oof_Ineffective
0,0,629862A6AB15,hould be brought t,2,None,-1,0.275358,0.717731,0.006911,i think cell phone should be brought to school...,20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 3...,Claim,0.717731,0.006911,0.275358
1,1,629862A6AB15,school and allowed to,2,None,-1,0.533104,0.464270,0.002625,i think cell phone should be brought to school...,39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 5...,Claim,0.464270,0.002625,0.533104
2,2,629862A6AB15,be used during lunch,3,None,-1,0.510031,0.486353,0.003616,i think cell phone should be brought to school...,62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 7...,Evidence,0.486353,0.003616,0.510031


In [78]:
oof_cols

['oof_Adequate', 'oof_Effective', 'oof_Ineffective']

In [79]:
class FeedbackStackerModel(nn.Module):
    def __init__(self, n_features, n_classes=3):
        super(FeedbackStackerModel, self).__init__()

        self.sizes = [256, 128, 64]
        print('num meta feats:', n_features)

        self.features = nn.Sequential(
            nn.utils.weight_norm(nn.Linear(n_features, self.sizes[0])),
            nn.PReLU(),
            nn.Linear(self.sizes[0], self.sizes[1]),
            nn.PReLU(),
            nn.Linear(self.sizes[1], self.sizes[2]),
            nn.PReLU(),
        )
        self.head = nn.Linear(self.sizes[-1], n_classes)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x, y=None):
        x = self.features(x)
        x = self.head(x)

        return {"logits": x}


class FeedbackStackerDataset(Dataset):
    def __init__(self, df, mode):
        self.df = df.copy().reset_index(drop=True)
        self.mode = mode

        self.feature_cols = oof_cols.copy()
        self.label_cols = label_cols.copy()

        df = self.df

        df["len"] = df.groupby("essay_id")[f"discourse_id"].transform("count") / 10
        self.feature_cols.append("len")

        for j, l in enumerate(label_cols):
            df[f"oof_{l}_mean"] = df.groupby("essay_id")[f"oof_{l}"].transform("mean")
            self.feature_cols.append(f"oof_{l}_mean")

            df[f"oof_{l}_t_mean"] = df.groupby(["essay_id", "discourse_type"])[
                f"oof_{l}"
            ].transform("mean")
            self.feature_cols.append(f"oof_{l}_t_mean")

        self.num_features = len(self.feature_cols)

        self.X = self.df[self.feature_cols].values
        #self.y = self.df[self.label_cols].values
        self.y = np.zeros((len(self.df),3))

    def __getitem__(self, idx):
        X = self.X[idx]
        y = self.y[idx]

        return torch.FloatTensor(X), torch.FloatTensor(y)

    def __len__(self):
        return self.df.shape[0]


import collections

def run_nn_stacker(exp_name, df, BS=64, device='cuda'):
    ds = FeedbackStackerDataset(df.iloc[:].copy(), mode="test")

    checkpoints = glob(f"{exp_name}/*.pth")

    models = []
    for checkpoint in checkpoints:
        print(f"running model {checkpoint}")

        model = FeedbackStackerModel(n_features=ds.num_features).to(device).eval()
        model_weights = torch.load(checkpoint, map_location=device)

        model.load_state_dict(collections.OrderedDict(model_weights), strict=True)
        models.append(model)

    dl = DataLoader(ds, shuffle=False, batch_size=BS, num_workers=4)

    with torch.no_grad():
        preds = []
        for batch in dl:
            data = [x.to(device) for x in batch]
            inputs, target = data
            p = []
            for model in models:
                out = model(inputs, target)
                p.append(out["logits"].float().softmax(dim=1))
            preds.append(torch.mean(torch.stack(p), dim=0).detach().cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    print('predictions:', preds.shape)
    return preds

In [80]:
# ds = FeedbackStackerDataset(df.copy(), mode="val")
# ds[0][0].shape, ds[0][1].shape

In [81]:
# nn_stacker_preds = run_nn_stacker("../input/efficiency-prize-v2/nn", df, BS=256)
nn_stacker_preds = run_nn_stacker("../input/efficiency-prize-v2/nn", df, BS=256, device='cpu')

running model ../input/efficiency-prize-v2/nn/checkpoint_fold2_seed0.pth
num meta feats: 10
running model ../input/efficiency-prize-v2/nn/checkpoint_fold1_seed0.pth
num meta feats: 10
running model ../input/efficiency-prize-v2/nn/checkpoint_fold4_seed0.pth
num meta feats: 10
running model ../input/efficiency-prize-v2/nn/checkpoint_fold3_seed0.pth
num meta feats: 10
running model ../input/efficiency-prize-v2/nn/checkpoint_fold0_seed0.pth
num meta feats: 10
predictions: (90974, 3)


In [82]:
preds_df_nn = pd.DataFrame(nn_stacker_preds, columns=label_cols)
preds_df_nn

,Adequate,Effective,Ineffective
0,0.647872,0.003284,0.348844
1,0.484868,0.003564,0.511568
2,0.495704,0.003234,0.501062
3,0.551172,0.002778,0.446050
4,0.651339,0.003429,0.345231
...,...,...,...
90969,0.536585,0.005505,0.457910
90970,0.648289,0.004482,0.347229
90971,0.538091,0.005267,0.456642
90972,0.548901,0.004437,0.446663


In [ ]:
# df["Adequate"] = all_preds[:, 0]
# df["Effective"] = all_preds[:, 1]
# df["Ineffective"] = all_preds[:, 2]

In [ ]:
# preds_nn = np.zeros(orig_preds.shape)
# preds_nn[:,0] = nn_stacker_preds[:, 2]  ## "Ineffective"
# preds_nn[:,1] = nn_stacker_preds[:, 0]  ## "Adequate"
# preds_nn[:,2] = nn_stacker_preds[:, 1]  ## "Effective"

In [ ]:
## ENSEMBLE ORIG + NN

# all_preds = [
#     orig_preds,
#     nn_preds,
# ]

# all_preds = np.average(all_preds, axis=0, weights=[2, 1])
# all_preds.shape

In [ ]:
# import matplotlib.pyplot as plt 
# import seaborn as sns 

# for col in label_cols:
#     sns.histplot(preds_df[col])
#     sns.histplot(preds_df_nn[col], color='r')
#     plt.show()

In [83]:
preds_df_ens1 = preds_df.copy()
for col in label_cols:
    preds_df_ens1[col] = 0.75*preds_df[col].values + 0.25*preds_df_nn[col].values

preds_df_ens1

,Ineffective,Adequate,Effective
0,0.293730,0.700266,0.006004
1,0.527720,0.469420,0.002860
2,0.507789,0.488691,0.003521
3,0.443150,0.554212,0.002638
4,0.329367,0.667642,0.002991
...,...,...,...
90969,0.462155,0.532899,0.004945
90970,0.306291,0.687150,0.006558
90971,0.460598,0.535198,0.004204
90972,0.440348,0.556095,0.003556


In [84]:
## prepare submision

sub3 = pd.read_csv('sub_02.csv')
print('num ids:', sub3.essay_id.nunique())
sub3 = sub3.rename(columns={'essay_id': 'essay_id_comp'})

# ## token types
# sub3['discourse_type'] = sub3['discourse_type'].map(discourse_type_to_int) 
# ### convert predstring to str 
# sub3['predictionstring'] = sub3.apply(lambda x: " ".join(str(x) for x in eval(x.predictionstring)), axis=1)

### effectiveness 
sub3['score_discourse_effectiveness_0'] = preds_df_ens1['Ineffective'].values
sub3['score_discourse_effectiveness_1'] = preds_df_ens1['Adequate'].values + preds_df_ens1['Effective'].values

# sub3 = sub3[sample_sub.columns]

num ids: 10402


In [85]:
# sub_ens1 = pd.concat([sub3[sample_sub.columns], dummy_df], axis=0)
# sub_ens1 = sub_ens1.reset_index(drop=True)
sub3[sample_sub.columns].to_csv('submission_ens_nn_stack.csv')

In [86]:
sub3[sample_sub.columns]

,essay_id_comp,predictionstring,score_discourse_effectiveness_0,score_discourse_effectiveness_1,discourse_type
0,629862A6AB15,20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 3...,0.293730,0.706270,2
1,629862A6AB15,39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 5...,0.527720,0.472280,2
2,629862A6AB15,62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 7...,0.507789,0.492211,3
3,629862A6AB15,84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 9...,0.443150,0.556850,3
4,629862A6AB15,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19,0.329367,0.670633,1
...,...,...,...,...,...
90969,9FDFDC5EB16E,174 175 176 177 178 179 180 181 182 183 184 18...,0.462155,0.537845,3
90970,9FDFDC5EB16E,607 608 609 610 611 612 613 614 615 616 617 61...,0.306291,0.693709,3
90971,9FDFDC5EB16E,887 888 889 890 891 892 893 894 895 896 897 89...,0.460598,0.539402,3
90972,9FDFDC5EB16E,1166 1167 1168 1169 1170 1171 1172 1173 1174 1...,0.440348,0.559652,6


# LGB stack

In [87]:
def gen_x(values):
    return np.histogram(
        np.clip(values, 0.001, 0.999), bins=3, density=True, range=(0, 1)
    )[0]

In [88]:
gb = df.groupby("essay_id", sort=False)

df["n_types"] = gb["discourse_type"].transform(lambda x: x.nunique())
df["mean_Ineffective"] = gb["Ineffective"].transform("mean")

all_groups = []
for name, group in gb:
    class_name = "Ineffective"
    for idx, val in enumerate(gen_x(group[class_name].values)):
        group[f"{class_name}_bin_{idx}"] = val

    all_groups.append(group)
df = pd.concat(all_groups).reset_index(drop=True)

disc_types_mapping = {
    "Lead": 0,
    "Position": 1,
    "Claim": 2,
    "Evidence": 3,
    "Counterclaim": 4,
    "Rebuttal": 5,
    "Concluding Statement": 6,
}

df["len_disc"] = df.discourse_text.str.len()
df["discourse_type"] = df["discourse_type"].map(disc_types_mapping)
df["paragraph_cnt"] = df.text.map(lambda x: len(x.split("\n\n")))

In [89]:
df.head()

,discourse_id,essay_id,discourse_text,discourse_type,discourse_effectiveness,fold,Ineffective,Adequate,Effective,text,...,oof_Adequate,oof_Effective,oof_Ineffective,n_types,mean_Ineffective,Ineffective_bin_0,Ineffective_bin_1,Ineffective_bin_2,len_disc,paragraph_cnt
0,0,629862A6AB15,hould be brought t,NaN,None,-1,0.275358,0.717731,0.006911,i think cell phone should be brought to school...,...,0.717731,0.006911,0.275358,4,0.458871,1.0,1.5,0.5,18,1
1,1,629862A6AB15,school and allowed to,NaN,None,-1,0.533104,0.464270,0.002625,i think cell phone should be brought to school...,...,0.464270,0.002625,0.533104,4,0.458871,1.0,1.5,0.5,22,1
2,2,629862A6AB15,be used during lunch,NaN,None,-1,0.510031,0.486353,0.003616,i think cell phone should be brought to school...,...,0.486353,0.003616,0.510031,4,0.458871,1.0,1.5,0.5,21,1
3,3,629862A6AB15,eriods and free times because if we couldn't s...,NaN,None,-1,0.442183,0.555225,0.002592,i think cell phone should be brought to school...,...,0.555225,0.002592,0.442183,4,0.458871,1.0,1.5,0.5,52,1
4,4,629862A6AB15,i think cell phone,NaN,None,-1,0.324078,0.673077,0.002845,i think cell phone should be brought to school...,...,0.673077,0.002845,0.324078,4,0.458871,1.0,1.5,0.5,19,1


In [90]:
lgb_metafeats = [
    "discourse_type",
    "Adequate",
    "Effective",
    "Ineffective",
    "n_types",
    "Ineffective_bin_0",
    "Ineffective_bin_1",
    "Ineffective_bin_2",
    "mean_Ineffective",
    "len_disc",
    "paragraph_cnt",
]

In [91]:
df[lgb_metafeats].shape

(90974, 11)

In [92]:
import lightgbm

lgb_stacker_preds = []
for fold in tqdm(range(5)):
    gbm = lightgbm.Booster(model_file=f"../input/efficiency-prize-v2/lightgbm/model_fold_{fold}.txt")
    valid_pred = gbm.predict(df[lgb_metafeats])
    lgb_stacker_preds.append(valid_pred)
    
preds_lgb = np.array(lgb_stacker_preds).mean(axis=0)
preds_lgb.shape

  0%|          | 0/5 [00:00<?, ?it/s]

(90974, 3)

In [93]:
preds_df_lgb = pd.DataFrame(preds_lgb, columns=label_cols)
preds_df_lgb

,Adequate,Effective,Ineffective
0,0.663390,0.005914,0.330696
1,0.456480,0.000384,0.543136
2,0.459858,0.002629,0.537513
3,0.587901,0.000347,0.411752
4,0.649410,0.000333,0.350258
...,...,...,...
90969,0.333270,0.004628,0.662102
90970,0.505880,0.005035,0.489085
90971,0.332339,0.001824,0.665836
90972,0.408486,0.000346,0.591168


In [ ]:
# all_preds = [
#     orig_preds,
#     lgb_stacker_preds,
#     nn_stacker_preds,
# ]

# all_preds = np.average(all_preds, axis=0, weights=[2, 1, 1 ])

In [94]:
label_cols

['Adequate', 'Effective', 'Ineffective']

In [95]:
preds_df_ens2 = preds_df.copy()
for col in label_cols:
    preds_df_ens2[col] = 0.75*preds_df[col].values + 0.25*preds_df_lgb[col].values

preds_df_ens2

,Ineffective,Adequate,Effective
0,0.289193,0.704145,0.006662
1,0.535612,0.462323,0.002065
2,0.516901,0.479729,0.003370
3,0.434575,0.563394,0.002031
4,0.330623,0.667160,0.002217
...,...,...,...
90969,0.513204,0.482071,0.004726
90970,0.341755,0.651548,0.006697
90971,0.512897,0.483760,0.003343
90972,0.476475,0.520992,0.002534


In [99]:
## prepare submision
sub = pd.read_csv('sub_02.csv')
print('num ids:', sub.essay_id.nunique())
sub = sub.rename(columns={'essay_id': 'essay_id_comp'})

# ## token types
# sub['discourse_type'] = sub['discourse_type'].map(discourse_type_to_int) 
# ### convert predstring to str 
# sub['predictionstring'] = sub.apply(lambda x: " ".join(str(x) for x in eval(x.predictionstring)), axis=1)

### effectiveness 
sub['score_discourse_effectiveness_0'] = preds_df_ens2['Ineffective'].values
sub['score_discourse_effectiveness_1'] = preds_df_ens2['Adequate'].values + preds_df_ens2['Effective'].values

# if len(missing_ids)>0:
#     sub = pd.concat([sub[sample_sub.columns], dummy_df], axis=0)
#     sub = sub.reset_index(drop=True)
# else:
#     sub = sub.reset_index(drop=True)
### SUBMIT 
sub[sample_sub.columns].to_csv('submission_v6_ens_lgb_stack.csv', index=False)

num ids: 10402


In [100]:
sub[sample_sub.columns]

,essay_id_comp,predictionstring,score_discourse_effectiveness_0,score_discourse_effectiveness_1,discourse_type
0,629862A6AB15,20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 3...,0.289193,0.710807,2
1,629862A6AB15,39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 5...,0.535612,0.464388,2
2,629862A6AB15,62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 7...,0.516901,0.483099,3
3,629862A6AB15,84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 9...,0.434575,0.565425,3
4,629862A6AB15,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19,0.330623,0.669377,1
...,...,...,...,...,...
90969,9FDFDC5EB16E,174 175 176 177 178 179 180 181 182 183 184 18...,0.513204,0.486796,3
90970,9FDFDC5EB16E,607 608 609 610 611 612 613 614 615 616 617 61...,0.341755,0.658245,3
90971,9FDFDC5EB16E,887 888 889 890 891 892 893 894 895 896 897 89...,0.512897,0.487103,3
90972,9FDFDC5EB16E,1166 1167 1168 1169 1170 1171 1172 1173 1174 1...,0.476475,0.523525,6


In [101]:
## ENSEMBLE ORIG + NN + LGB 

preds_df_ens3 = preds_df.copy()
for col in label_cols:
    preds_df_ens3[col] = 0.5*preds_df[col].values + 0.25*preds_df_lgb[col].values + 0.25*preds_df_nn[col].values

preds_df_ens3

,Ineffective,Adequate,Effective
0,0.307564,0.686681,0.005755
1,0.530228,0.467472,0.002300
2,0.514659,0.482067,0.003274
3,0.435542,0.562381,0.002077
4,0.335911,0.661726,0.002363
...,...,...,...
90969,0.511788,0.483299,0.004912
90970,0.355401,0.638594,0.006005
90971,0.511578,0.484724,0.003698
90972,0.478579,0.518593,0.002827


In [102]:
## prepare submision
sub = pd.read_csv('sub_02.csv')
print('num ids:', sub.essay_id.nunique())

sub = sub.rename(columns={'essay_id': 'essay_id_comp'})
# ## token types
# sub['discourse_type'] = sub['discourse_type'].map(discourse_type_to_int) 
# ### convert predstring to str 
# sub['predictionstring'] = sub.apply(lambda x: " ".join(str(x) for x in eval(x.predictionstring)), axis=1)

### effectiveness 
sub['score_discourse_effectiveness_0'] = preds_df_ens3['Ineffective'].values
sub['score_discourse_effectiveness_1'] = preds_df_ens3['Adequate'].values + preds_df_ens2['Effective'].values

# if len(missing_ids)>0:
#     sub = pd.concat([sub[sample_sub.columns], dummy_df], axis=0)
#     sub = sub.reset_index(drop=True)
# else:
#     sub = sub.reset_index(drop=True)
### SUBMIT 
sub[sample_sub.columns].to_csv('submission_v6_ens_stack.csv', index=False)

num ids: 10402


In [105]:
sub[sample_sub.columns].head()

,essay_id_comp,predictionstring,score_discourse_effectiveness_0,score_discourse_effectiveness_1,discourse_type
0,629862A6AB15,20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 3...,0.307564,0.693343,2
1,629862A6AB15,39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 5...,0.530228,0.469537,2
2,629862A6AB15,62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 7...,0.514659,0.485436,3
3,629862A6AB15,84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 9...,0.435542,0.564412,3
4,629862A6AB15,0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19,0.335911,0.663942,1


In [106]:
%ls -l *.csv

-rw-r--r-- 1 root root 251374460 Mar 12 09:39 sub_02.csv
-rw-r--r-- 1 root root  20064525 Mar 12 09:42 submission_ens_nn_stack.csv
-rw-r--r-- 1 root root  20065182 Mar 12 09:36 submission_rater_full.csv
-rw-r--r-- 1 root root  19529815 Mar 12 09:45 submission_v6_ens_lgb_stack.csv
-rw-r--r-- 1 root root  19529875 Mar 12 09:46 submission_v6_ens_stack.csv
